# 01. The three question paradigms

There are three ways to ask a decision model a question. Noul returns the probability that a statement is true. Choice returns a probability for each option we declare. Score returns a probability for each level of an ordered rubric.

In this notebook we build each one, inspect its declared answer labels, and validate the results. Every code cell does one small thing.


<div align='center'><img src='../images/question_types.png' alt='The three question types' style='width:80%; max-width:850px;'></div>

*The same state can be queried in three ways. The declared answers differ, but the output is always one probability per answer.*


## Step 1. Noul asks about truth

A Noul question contains one statement. The declared answers are always `true` and `false`, so the model cannot invent a third option. We start with a true statement.


In [ ]:
from decision_qa import MockClient, NoulQuestion, question_labels
client = MockClient(seed=1)


In [ ]:
q_true = NoulQuestion(label='fact', statement='Paris is the capital of France.', gold=True)
print(q_true.statement)


In [ ]:
labels = question_labels(q_true)
print(labels)
assert labels == ['true', 'false']


In [ ]:
res_true = client.decision({'question': ' geography '}, [q_true])[0]
print(res_true.probabilities)


In [ ]:
assert res_true.probabilities['true'] > res_true.probabilities['false']
print('true statement ranked first')


Now we repeat the exercise with a false statement. The declared labels are unchanged; only the statement text changes. This shows that the paradigm is fixed while the input varies.


In [ ]:
q_false = NoulQuestion(label='fact2', statement='Rome is the capital of Germany.', gold=False)
res_false = client.decision({}, [q_false])[0]
print(res_false.probabilities)


In [ ]:
assert res_false.probabilities['false'] > res_false.probabilities['true']
print('false statement ranked last')


## Step 2. Choice declares its options

Choice is the natural fit for classification. We supply a list of options and the model returns one probability per option. The `gold_index` is metadata for evaluation.


In [ ]:
from decision_qa import ChoiceQuestion
q_choice = ChoiceQuestion(label='color', options=['red', 'green', 'blue'], gold_index=1)
print(q_choice.options)


In [ ]:
choice_labels = question_labels(q_choice)
print(choice_labels)
assert choice_labels == q_choice.options


In [ ]:
res_choice = client.decision({}, [q_choice])[0]
print(res_choice.probabilities)


## Step 3. Shuffling options without losing the answer

A model may be sensitive to position. To test that, we shuffle the options and must carry the gold index along. `shuffle_with_gold` returns both the new list and the new index.


In [ ]:
import random
from decision_qa.distractors import shuffle_with_gold
rng = random.Random(0)
shuffled, new_gold = shuffle_with_gold(q_choice.options, q_choice.gold_index, rng)
print(shuffled, new_gold)


In [ ]:
assert shuffled[new_gold] == q_choice.options[q_choice.gold_index]
print('gold option preserved')


## Step 4. Score uses an ordered rubric

Score differs from Choice because the levels have an order. Here `poor`, `fair`, and `good` are not interchangeable. We print them and confirm their order is unchanged.


In [ ]:
from decision_qa import ScoreQuestion
q_score = ScoreQuestion(label='quality', rubric='Rate answer quality.', levels=['poor', 'fair', 'good'], gold_index=2)
print(q_score.levels)


In [ ]:
assert q_score.levels == ['poor', 'fair', 'good']
print('rubric order preserved')


In [ ]:
res_score = client.decision({}, [q_score])[0]
print(res_score.probabilities)


## Step 5. Confidence and abstention

A probability is useful only if we know when to trust it. A simple policy commits when the top probability clears a threshold and escalates otherwise. Escalation can mean calling a larger model or asking a person.


In [ ]:
def act(result, threshold=0.8):
    label = max(result.probabilities, key=result.probabilities.get)
    p = result.probabilities[label]
    return label, 'commit' if p >= threshold else 'escalate'


In [ ]:
print(act(res_true))


In [ ]:
action, decision = act(res_true)
assert decision in ('commit', 'escalate')
print('policy returned a valid action')


In [ ]:
print(act(res_true, threshold=0.99))


The dashed line is the confidence threshold. Everything above it can be committed; everything below it should be escalated. The plot makes the decision rule visible.


In [ ]:
import matplotlib.pyplot as plt
labels = list(res_true.probabilities)
values = list(res_true.probabilities.values())
fig, ax = plt.subplots(figsize=(6.4, 3.4), constrained_layout=True)
ax.bar(labels, values, color='#4C78A8')
ax.axhline(0.8, color='#E45756', linestyle='--', label='threshold 0.8')
ax.set_ylim(0, 1.1)
ax.set_ylabel('Probability', labelpad=8)
ax.set_title('Noul result with a confidence gate')
ax.grid(axis='y', alpha=0.25)
ax.legend()
for index, value in enumerate(values):
    ax.text(index, value + 0.03, f'{value:.2f}', ha='center', fontsize=9)
plt.show()


## Step 6. Batch several questions

One request can carry many questions. The client returns one result per question, in the same order. We verify that the order is preserved.


In [ ]:
batch = client.decision({}, [q_true, q_choice, q_score])
print([r.question_label for r in batch])


In [ ]:
assert [r.question_label for r in batch] == ['fact', 'color', 'quality']
print('result order matches request order')


## Step 7. Option order invariance

A reliable decision should not change when we reorder the options. We measure this with `option_order_invariance`. The mock is biased by the gold label, so it should be invariant.


In [ ]:
from decision_qa.evaluate import option_order_invariance
invariant = option_order_invariance(client, {}, q_choice)
print(invariant)


In [ ]:
assert invariant is True
print('option order did not change the top answer')


## Common mistakes

First, never ask a Score question with shuffled levels. Second, never omit options from a Choice question, because the model only scores what we declare. Third, always define a confidence policy before putting a decision model in production.

Free further reading: `awesome-jev-prompts` has 51 field-tested question patterns under CC0, and the concept docs introduce the same three paradigms.
